In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torchvision.models import VGG16_Weights

In [2]:
#i suppose gt boxes format is x1,y1,x2,y2
import math as mt 
def iou(x1A,y1A,x2A,y2A,x1B,y1B,x2B,y2B):
    a=min(x2A,x2B)
    b=max(x1B,x1A)
    d=max(y1A,y1B)
    c=min(y2A,y2B)

    h=max(0,a-b)
    w=max(0,c-d)
    intersection=h*w

    A_area=(x2A-x1A)*(y2A-y1A)
    B_area=(x2B-x1B)*(y2B-y1B)

    union=A_area+B_area-intersection

    return intersection/union
def calculate_sk(k,m):
    """
    k is index of ft map , m is total nb of ft map 
    (ft map are map participating in prediction)
    k=1....M
    """
    smin,smax=0.2,0.9
    if k>m:
        return 1
    return smin+(smax-smin)*(k-1)/(m-1)


def calculate_anchor_w_h1(sk,a):
    return sk*mt.sqrt(a),sk/mt.sqrt(a)

def calculate_anchor_w_h2(sk,a):
    return sk/mt.sqrt(a),sk*mt.sqrt(a)


def normalised_anchor_coords(i,j,f,w,h):

    """
    i , j in len(feature map)=0...f-1
    w=w_k_a
    h=h_k_a
    """
    centerx=(j+0.5)/f
    centery=(i+0.5)/f


    x2=min(centerx+w/2,1)
    y2=min(centery+h/2,1)

    x1=max(centerx-w/2,0)
    y1=max(centery-h/2,0)
    return x1,y1,x2,y2


def normalised_gt_coords(x1,y1,x2,y2,H,W):
    """
    normalise gt coords so that they are in [0,1]
    """
    return x1/W,y1/H,x2/W,y2/H

def corner_to_center(x1, y1, x2, y2):
    w = x2 - x1
    h = y2 - y1
    cx = x1 + w / 2
    cy = y1 + h / 2
    return cx, cy, w, h


def center_to_corner(cx, cy, w, h):
    x1 = cx - w / 2
    y1 = cy - h / 2
    x2 = cx + w / 2
    y2 = cy + h / 2
    return x1, y1, x2, y2




In [147]:
vgg = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
vgg[16] = nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)

In [148]:
base=nn.ModuleList(vgg[:30])#until 5_3 layer

In [149]:
base
convs={}
i=2
convid=1
print("----","conv",1,"----")

for id, el in enumerate(base):

    print(f"({id})",el )
    if isinstance(el,nn.Conv2d):
        
        convs[id]=f"{i}_{convid}"
        convid=convid+1
    if isinstance(el,nn.MaxPool2d):
        convid=1
        print("----","conv",i,"----")
        i=i+1
    

    

    
    

---- conv 1 ----
(0) Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(1) ReLU(inplace=True)
(2) Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(3) ReLU(inplace=True)
(4) MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
---- conv 2 ----
(5) Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(6) ReLU(inplace=True)
(7) Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(8) ReLU(inplace=True)
(9) MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
---- conv 3 ----
(10) Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(11) ReLU(inplace=True)
(12) Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(13) ReLU(inplace=True)
(14) Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(15) ReLU(inplace=True)
(16) MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
---- conv 4 ----
(17) Conv2d(256, 512, kernel_s

In [ ]:
from l2norm import L2norm
class SSD(nn.Module):
    def __init__(self,base,nb_classes):
        super().__init__()
        self.features=base
        self.nb_classes=nb_classes

        self.l2norm=L2norm(512,20)

        self.extras=nn.ModuleList([
            #conv6 and conv7
            nn.Sequential(
                nn.Conv2d(in_channels=512,out_channels=1024,kernel_size=3,padding=6,dilation=6),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels=1024,out_channels=1024,kernel_size=1),
                nn.ReLU(inplace=True),    
            ),

            #conv8_2
            nn.Sequential(
                nn.Conv2d(in_channels=1024,out_channels=256,kernel_size=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels=256,out_channels=512,kernel_size=3,padding=1,stride=2),
                nn.ReLU(inplace=True),    
            ),

            #conv9_2
            nn.Sequential(
                nn.Conv2d(in_channels=512,out_channels=128,kernel_size=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels=128,out_channels=256,kernel_size=3,padding=1,stride=2),
                nn.ReLU(inplace=True),    
            ),

            #conv10_2
            nn.Sequential(
                nn.Conv2d(in_channels=256,out_channels=128,kernel_size=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels=128,out_channels=256,kernel_size=3),
                nn.ReLU(inplace=True),    
            ),

            #conv11_2
            nn.Sequential(
                nn.Conv2d(in_channels=256,out_channels=128,kernel_size=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels=128,out_channels=256,kernel_size=3),
                nn.ReLU(inplace=True),    
            )
        ])

        #define kernels for classification to output Feature Map of sieze H,W,ki*nb_classes for all H,W, ki in {4,6} for all i =1 ... |ssd feature maps | , ki is number of anchors for each location of H,W, image 
        self.classification_convolutions=nn.ModuleList([

            nn.Conv2d(512,4*nb_classes,kernel_size=3,padding=1),
            nn.Conv2d(1024,6*nb_classes,kernel_size=3,padding=1),
            nn.Conv2d(512,6*nb_classes,kernel_size=3,padding=1),
            nn.Conv2d(256,6*nb_classes,kernel_size=3,padding=1),
            nn.Conv2d(256,4*nb_classes,kernel_size=3,padding=1),
            nn.Conv2d(256,4*nb_classes,kernel_size=3,padding=1),
        ])

        #same but using 4 coordinates for each anchor 
        self.regression_convolutions=nn.ModuleList([

            nn.Conv2d(512,4*4,kernel_size=3,padding=1),
            nn.Conv2d(1024,6*4,kernel_size=3,padding=1),
            nn.Conv2d(512,6*4,kernel_size=3,padding=1),
            nn.Conv2d(256,6*4,kernel_size=3,padding=1),
            nn.Conv2d(256,4*4,kernel_size=3,padding=1),
            nn.Conv2d(256,4*4,kernel_size=3,padding=1),

        ])
    def forward(self, X):
        layers_for_prediction = []
      
        #base model 
        for idx in range(len(self.features)):

            X=self.features[idx](X)
            
            if idx in convs and convs[idx]=="4_3":

                X=self.l2norm(X)
                layers_for_prediction.append(X)
          
                
        for idx in range(len(self.extras)):
            X=self.extras[idx](X)
            
            layers_for_prediction.append(X)


        classifications=[]    
        for layer_for_predictions, classification_convolution in zip( layers_for_prediction, self.classification_convolutions):
            classifications.append(classification_convolution(layer_for_predictions))
        
        regressions=[]  
        for layer_for_predictions, regression_convolution in zip( layers_for_prediction, self.regression_convolutions):
            regressions.append(regression_convolution(layer_for_predictions))

        return classifications,regressions,layers_for_prediction



In [38]:
convs

{0: '1_1',
 2: '1_2',
 5: '2_1',
 7: '2_2',
 10: '3_1',
 12: '3_2',
 14: '3_3',
 17: '4_1',
 19: '4_2',
 21: '4_3',
 24: '5_1',
 26: '5_2',
 28: '5_3'}

In [34]:
layers=[]
pool5 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
conv6 = nn.Conv2d(512, 1024, kernel_size=3, padding=6, dilation=6)
conv7 = nn.Conv2d(1024, 1024, kernel_size=1)
layers += [pool5, conv6,
               nn.ReLU(inplace=True), conv7, nn.ReLU(inplace=True)]
layers[1]

Conv2d(512, 1024, kernel_size=(3, 3), stride=(1, 1), padding=(6, 6), dilation=(6, 6))

In [29]:
X=torch.randn((10,3,300,300))

model=SSD(base,21)
classifications,regressions,layers_for_prediction=model.forward(X)

In [30]:
for el  in layers_for_prediction:
    print(el.shape[2:])



torch.Size([38, 38])
torch.Size([19, 19])
torch.Size([10, 10])
torch.Size([5, 5])
torch.Size([3, 3])
torch.Size([1, 1])
